# LLM API Comparison: DeepSeek, Gemma (Vision) & Mistral
This notebook demonstrates:
- **DeepSeek R1** via HuggingFace Router (text reasoning)
- **Gemma 4** via HuggingFace Router (multimodal: text + image)
- **Mistral** via Mistral API (text chat)

All API keys are centralised in the Configuration cell below.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
# Store API keys here using environment variables — never hardcode secrets.
import os
import requests

HF_API_KEY      = os.getenv("HF_API_KEY",      "YOUR_HUGGINGFACE_API_KEY_HERE")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY_HERE")

HF_ROUTER_URL = "https://router.huggingface.co/v1/chat/completions"
MISTRAL_URL   = "https://api.mistral.ai/v1/chat/completions"

REQUEST_TIMEOUT = 60  # seconds (DeepSeek R1 can be slow due to chain-of-thought)

print("Configuration loaded.")

In [ ]:
# ─── Shared HuggingFace Router Helper ────────────────────────────────────────
def query_hf_router(messages: list, model: str) -> str:
    """
    Generic helper for any model served via HuggingFace Router
    using the OpenAI-compatible /v1/chat/completions endpoint.

    Args:
        messages: OpenAI-style message list, e.g.
                  [{"role": "user", "content": "Hello"}]
        model:    HuggingFace model ID (with optional :provider suffix)

    Returns:
        The assistant's reply as a string, or an error message.
    """
    headers = {
        "Authorization": f"Bearer {HF_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {"messages": messages, "model": model}
    try:
        response = requests.post(HF_ROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.HTTPError as e:
        return f"[HF Router HTTP Error] {e} — {response.text}"
    except requests.exceptions.Timeout:
        return "[HF Router Error] Request timed out."
    except (KeyError, IndexError):
        return f"[HF Router Error] Unexpected response: {response.json()}"
    except Exception as e:
        return f"[HF Router Error] {e}"


print("HF Router helper ready.")

In [ ]:
# ─── DeepSeek R1 (Text Reasoning) ────────────────────────────────────────────
def query_deepseek(prompt: str) -> str:
    """
    Query DeepSeek-R1 via HuggingFace Router.
    This model uses chain-of-thought (<think>...</think>) before answering.
    """
    messages = [{"role": "user", "content": prompt}]
    raw = query_hf_router(messages, model="deepseek-ai/DeepSeek-R1-0528:novita")

    # Strip the internal <think>...</think> block for a cleaner display
    if "<think>" in raw and "</think>" in raw:
        think_end = raw.find("</think>")
        return raw[think_end + len("</think>"):].strip()
    return raw


# Test
answer = query_deepseek("What is the capital of Afghanistan?")
print("DeepSeek R1 response:\n")
print(answer)

In [ ]:
# ─── Gemma 4 Vision (Text + Image) ───────────────────────────────────────────
def query_gemma_vision(text_prompt: str, image_url: str) -> str:
    """
    Query Gemma 4 with a combined text + image input via HuggingFace Router.

    Args:
        text_prompt: The instruction about the image.
        image_url:   Publicly accessible URL of the image.

    Returns:
        The model's description/answer as a string.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text",      "text": text_prompt},
                {"type": "image_url", "image_url": {"url": image_url}}
            ]
        }
    ]
    return query_hf_router(messages, model="google/gemma-4-31B-it:novita")


# Test
image = "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
description = query_gemma_vision("Describe this image in one sentence.", image)
print("Gemma Vision response:\n")
print(description)

In [ ]:
# ─── Mistral API (Text Chat) ──────────────────────────────────────────────────
def query_mistral(prompt: str, model: str = "mistral-small") -> str:
    """
    Query Mistral's chat completion API.

    Returns:
        The assistant's reply as a string, or an error message.
    """
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}]
    }
    try:
        response = requests.post(MISTRAL_URL, json=payload, headers=headers, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.HTTPError as e:
        return f"[Mistral HTTP Error] {e} — {response.text}"
    except requests.exceptions.Timeout:
        return "[Mistral Error] Request timed out."
    except (KeyError, IndexError):
        return f"[Mistral Error] Unexpected response: {response.json()}"
    except Exception as e:
        return f"[Mistral Error] {e}"


# Test
answer = query_mistral("What is the meaning of life?")
print("Mistral response:\n")
print(answer)

In [ ]:
# ─── Text Model Comparison ────────────────────────────────────────────────────
def compare_text_models(prompt: str):
    """
    Compare DeepSeek R1 and Mistral on the same text prompt.
    """
    print(f"Prompt: {prompt!r}")
    print("=" * 60)

    for name, fn in [("DeepSeek R1", query_deepseek), ("Mistral (mistral-small)", query_mistral)]:
        print(f"\n[{name}]")
        print("-" * 40)
        print(fn(prompt))


compare_text_models("Explain deep learning in simple terms.")